#### Dependencies

In [ ]:
import os

from azure.ai.ml import MLClient
from azure.ai.ml.entities import AmlCompute, Environment, CommandJob
from azure.identity import DefaultAzureCredential, ManagedIdentityCredential
from azureml.core import Workspace, Experiment, Datastore

#### Create Handle

In [ ]:
# obtain workspace auth details from local config file
ws = Workspace.from_config()

# choose credential access route
credential = ManagedIdentityCredential()
#credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=ws.subscription_id,
    resource_group_name=ws.resource_group,
    workspace_name=ws.name
)

#### Create YAML file

In [ ]:
# create env folder if one doesn't exist
dependencies_dir = "./env"
os.makedirs(dependencies_dir, exist_ok=True)

#### Define Environment

In [ ]:
# Define environment
env = Environment(
    name="ray-env",
    description="Ray training environment to experiment running ray on AzureML",
    tags={"ray": "ray-torch"},
    conda_file=os.path.join(env_dir, "environment.yml"),
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
)

ml_client.environments.create_or_update(env)

#### Define Job

In [ ]:

# Define command job
job = CommandJob(
    code="./src",
    command="python train.py",
    environment="ray-env@latest",
    compute=compute_name,
    experiment_name="fashion-mnist-ray",
)

returned_job = ml_client.jobs.create_or_update(job)
print(returned_job.studio_url)